In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# RAG 절차
- data : https://law.go.kr/법령/소득세법 에서 doc/docx 다운로드, docx 포멧 통일

1. 문서를 읽는다 : python-docx 이용 pip install python-docx
2. 읽어온 문서를 쪼갠다 : chunks 분리, tiktoken 이용
    - 모델의 보편적인 context window가 128,000로 문서 내용이 많으면 한 번에 load 할 수 없어서 chunks 분리(한글의 경우 보통 1글자 1토큰)
    - 문서가 길면(문서 파일이 크면) ; input이 길면 비용과 시간이 오래 걸림
3. 쪼갠 문서를 임베딩 -> Vector DB에 저장 -> chroma(local vector DB), pinecorn(클라우드 vector DB)
4. 질문 query 과 vector DataBase의 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변을 생성

# 1. 문서를 읽는다 : python-docx 사용

In [2]:
%pip install python-docx

  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
Using cached python_docx-1.2.0-py3-none-any.whl (252 kB)
Note: you may need to restart the kernel to use updated packages.


In [18]:
from docx import Document

document = Document('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
print(dir(document)) # dir : 객체가 가지고 있는 속성을 체크

['_Document__body', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_block_width', '_body', '_element', '_parent', '_part', 'add_comment', 'add_heading', 'add_page_break', 'add_paragraph', 'add_picture', 'add_section', 'add_table', 'comments', 'core_properties', 'element', 'inline_shapes', 'iter_inner_content', 'paragraphs', 'part', 'save', 'sections', 'settings', 'styles', 'tables']


In [22]:
print(len(document.paragraphs))
for paragraph in document.paragraphs[:10] :
    print(paragraph.text)

3003
소득세법
[시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., 일부개정]
기획재정부(재산세제과(양도소득세)) 044-215-4312
기획재정부(소득세제과(근로소득)) 044-215-4216
기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4233
기획재정부(소득세제과(사업소득, 기타소득)) 044-215-4217

제1장 총칙 <개정 2009. 12. 31.>

제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다.


In [37]:
full_text = ""
for paragraph in document.paragraphs :
    full_text += paragraph.text + " "
print(full_text[:30])

소득세법 [시행 2025. 7. 1.] [법률 제206


In [38]:
print(full_text[:1500])

소득세법 [시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., 일부개정] 기획재정부(재산세제과(양도소득세)) 044-215-4312 기획재정부(소득세제과(근로소득)) 044-215-4216 기획재정부(금융세제과(이자소득, 배당소득)) 044-215-4233 기획재정부(소득세제과(사업소득, 기타소득)) 044-215-4217  제1장 총칙 <개정 2009. 12. 31.>  제1조(목적) 이 법은 개인의 소득에 대하여 소득의 성격과 납세자의 부담능력 등에 따라 적정하게 과세함으로써 조세부담의 형평을 도모하고 재정수입의 원활한 조달에 이바지함을 목적으로 한다. [본조신설 2009. 12. 31.] [종전 제1조는 제2조로 이동 <2009. 12. 31.>]  제1조의2(정의) ① 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2010. 12. 27., 2014. 12. 23., 2018. 12. 31.> 1. “거주자”란 국내에 주소를 두거나 183일 이상의 거소(居所)를 둔 개인을 말한다. 2. “비거주자”란 거주자가 아닌 개인을 말한다. 3. “내국법인”이란 「법인세법」 제2조제1호에 따른 내국법인을 말한다. 4. “외국법인”이란 「법인세법」 제2조제3호에 따른 외국법인을 말한다. 5. “사업자”란 사업소득이 있는 거주자를 말한다. ② 제1항에 따른 주소ㆍ거소와 거주자ㆍ비거주자의 구분은 대통령령으로 정한다. [본조신설 2009. 12. 31.]  제2조(납세의무) ① 다음 각 호의 어느 하나에 해당하는 개인은 이 법에 따라 각자의 소득에 대한 소득세를 납부할 의무를 진다. 1. 거주자 2. 비거주자로서 국내원천소득(國內源泉所得)이 있는 개인 ② 다음 각 호의 어느 하나에 해당하는 자는 이 법에 따라 원천징수한 소득세를 납부할 의무를 진다. 1. 거주자 2. 비거주자 3. 내국법인 4. 외국법인의 국내지점 또는 국내영업소(출장소, 그 밖에 이에 준하는 것을 포함한다. 이하 같다) 5. 그 밖에 이 법에서 정하는 원천징수의무자 ③

In [39]:
# docx 문서의 글자 및 paragraph 수
print(len(full_text))
print(len(document.paragraphs))

235201
3003


# 2. 읽어온 문서를 쪼갠다
- pip install tiktoken
- full_text -> token 단위로 쪼개서
- 1500토큰 씩 문서를 쪼개기

- %pip install tiktoken

In [40]:
%pip install tiktoken

Note: you may need to restart the kernel to use updated packages.


In [42]:
import tiktoken
# tiktoken 인식 가능한 모델 이름만 사용 가능 ex. gpt-4o시리즈, gpt-3.5-turbo, text-embedding-ada-002...
# gpt-4.1-nano 불가
encoder = tiktoken.encoding_for_model("gpt-4o-mini")
# 문자들 -> 숫자 리스트
encoding = encoder.encode(full_text)
encoding

[11226,
 64328,
 11734,
 22070,
 723,
 5637,
 15719,
 220,
 1323,
 20,
 13,
 220,
 22,
 13,
 220,
 16,
 54569,
 723,
 22070,
 73888,
 14377,
 20007,
 1055,
 16684,
 11,
 220,
 1323,
 19,
 13,
 220,
 899,
 13,
 220,
 2911,
 4213,
 124313,
 13915,
 7329,
 60,
 11061,
 60576,
 16012,
 180899,
 7,
 16012,
 17198,
 11734,
 8021,
 8562,
 7,
 21902,
 5827,
 11226,
 64328,
 11734,
 915,
 220,
 35132,
 12,
 21625,
 12,
 36948,
 17,
 11061,
 60576,
 16012,
 180899,
 7,
 11226,
 64328,
 11734,
 8021,
 8562,
 7,
 32481,
 3710,
 11226,
 64328,
 915,
 220,
 35132,
 12,
 21625,
 12,
 34096,
 21,
 11061,
 60576,
 16012,
 180899,
 7,
 16668,
 110223,
 11734,
 8021,
 8562,
 7,
 2186,
 5947,
 11226,
 64328,
 11,
 33628,
 19388,
 11226,
 64328,
 915,
 220,
 35132,
 12,
 21625,
 12,
 36645,
 18,
 11061,
 60576,
 16012,
 180899,
 7,
 11226,
 64328,
 11734,
 8021,
 8562,
 7,
 131266,
 11226,
 64328,
 11,
 168944,
 11226,
 64328,
 915,
 220,
 35132,
 12,
 21625,
 12,
 34096,
 22,
 220,
 14377,
 16,
 7681,
 64

In [43]:
print("full_text의 전체 토큰 수 :", len(encoding)) # gpt-4o-mini 기반 토큰 사이즈 체크

full_text의 전체 토큰 수 : 153699


In [50]:
# 숫자 리스트 -> 문자들
decoded = encoder.decode(encoding)
decoded[:50]

'소득세법 [시행 2025. 7. 1.] [법률 제20615호, 2024. 12. 31., '

In [51]:
print(decoded[:10])
print(encoding[:10])

소득세법 [시행 2
[11226, 64328, 11734, 22070, 723, 5637, 15719, 220, 1323, 20]


In [55]:
encoder.encode("소득세"), encoder.encode("안녕 친구"), encoder.encode("Hello"), encoder.encode("안녕")

([11226, 64328, 11734], [14307, 98931, 128075], [13225], [14307, 98931])

In [59]:
list(range(0,len(encoding),15000))

[0, 15000, 30000, 45000, 60000, 75000, 90000, 105000, 120000, 135000, 150000]

In [60]:
# full_text를 쪼개는 함수 : chunk_list 반환
import tiktoken
def split_text(full_text, chunk_size=1500) :
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    total_encoding = encoder.encode(full_text)
    chunk_list = []
    total_token_count = len(total_encoding)
    for i in range(0,total_token_count,chunk_size) :
        chunk = total_encoding[i:i+chunk_size]
        decoded = encoder.decode(chunk)
        chunk_list.append(decoded)
    return chunk_list

In [63]:
example_text = "소득세법 법률 일부 개정함 이자소득 배당소득"
split_text(example_text, 10)

['소득세법 법률 일부 개정함', ' 이자소득 배당소득']

In [64]:
chunk_list = split_text(full_text)

In [69]:
len(chunk_list)

103

# 3. 쪼갠 문서를 임베딩 -> DB에 저장
- chroma 공식 홈페이지 https://python.langchain.com/docs/integrations/vectorstores/chroma/
- pip install chromadb

In [70]:
%pip install chromadb

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using cached tomli-2.2.1-py3-none-any.whl.metadata (10 kB)
  Using cached google_auth-2.40.3-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached websocket_client-1.8.0-py3-none-any.whl.metadata (8.0 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [71]:
import chromadb
chroma_client = chromadb.Client()

In [74]:
# collection은 RDB에서의 Table 개념
collection_name = "tax_collection"

In [73]:
# 임베딩 객체
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
import os
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

openai_embedding = OpenAIEmbeddingFunction(
    model_name = "text-embedding-3-large",
    api_key=api_key,
)

In [87]:
# chroma db에 collection 생성
# tax_collection = chroma_client.create_collection(collection_name) # 컬렉션 이름만 지정
# chroma_client.delete_collection(collection_name)
tax_collection = chroma_client.get_or_create_collection(
    name = collection_name,
    embedding_function=openai_embedding
) 

In [83]:
# collection에 입력할 때 사용할 id들 : 문자
ids = [str(id) for id in range(len(chunk_list))]
print(ids)

['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102']


In [88]:
tax_collection.add(documents=chunk_list,
                  ids=ids)

# 4. 질문 query 과 Vector DataBase 유사도 검색

In [96]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_doc = tax_collection.query(query_texts=query,
                                    n_results=3, # 기본은 유사도가 높은 10개 추출
                                    )

In [97]:
retrieved_doc

{'ids': [['3', '4', '9']],
 'embeddings': None,
 'documents': [['하지 아니한다. <개정 2010. 12. 27., 2011. 7. 25., 2011. 9. 15., 2012. 2. 1., 2013. 1. 1., 2013. 3. 22., 2014. 1. 1., 2014. 3. 18., 2014. 12. 23., 2015. 12. 15., 2016. 12. 20., 2018. 3. 20., 2018. 12. 31., 2019. 12. 10., 2019. 12. 31., 2020. 6. 9., 2020. 12. 29., 2022. 8. 12., 2022. 12. 31., 2023. 8. 8., 2023. 12. 31., 2024. 12. 31.> 1. 「공익신탁법」에 따른 공익신탁의 이익 2. 사업소득 중 다음 각 목의 어느 하나에 해당하는 소득 가. 논ㆍ밭을 작물 생산에 이용하게 함으로써 발생하는 소득 나. 1개의 주택을 소유하는 자의 주택임대소득(제99조에 따른 기준시가가 12억원을 초과하는 주택 및 국외에 소재하는 주택의 임대소득은 제외한다) 또는 해당 과세기간에 대통령령으로 정하는 총수입금액의 합계액이 2천만원 이하인 자의 주택임대소득(2018년 12월 31일 이전에 끝나는 과세기간까지 발생하는 소득으로 한정한다). 이 경우 주택 수의 계산 및 주택임대소득의 산정 등 필요한 사항은 대통령령으로 정한다. 다. 대통령령으로 정하는 농어가부업소득 라. 대통령령으로 정하는 전통주의 제조에서 발생하는 소득 마. 조림기간 5년 이상인 임지(林地)의 임목(林木)의 벌채 또는 양도로 발생하는 소득으로서 연 600만원 이하의 금액. 이 경우 조림기간 및 세액의 계산 등 필요한 사항은 대통령령으로 정한다. 바. 대통령령으로 정하는 작물재배업에서 발생하는 소득 사. 대통령령으로 정하는 어로어업 또는 양식어업에서 발생하는 소득 3. 근로소득과 퇴직소득 중 다음 각 목의 어느 하나에 해당하는 소득 가. 대통령령으로 정하는 복무 중

In [103]:
retrieved_doc['documents'][0] # 중첩list
retrieved_doc['documents'][0][1]

'상대상자 지원에 관한 법률」에 따라 받는 보훈급여금ㆍ학습보조비 타. 「전직대통령 예우에 관한 법률」에 따라 받는 연금 파. 작전임무를 수행하기 위하여 외국에 주둔 중인 군인ㆍ군무원이 받는 급여 하. 종군한 군인ㆍ군무원이 전사(전상으로 인한 사망을 포함한다. 이하 같다)한 경우 그 전사한 날이 속하는 과세기간의 급여 거. 국외 또는 「남북교류협력에 관한 법률」에 따른 북한지역에서 근로를 제공하고 받는 대통령령으로 정하는 급여 너. 「국민건강보험법」, 「고용보험법」 또는 「노인장기요양보험법」에 따라 국가, 지방자치단체 또는 사용자가 부담하는 보험료 더. 생산직 및 그 관련 직에 종사하는 근로자로서 급여 수준 및 직종 등을 고려하여 대통령령으로 정하는 근로자가 대통령령으로 정하는 연장근로ㆍ야간근로 또는 휴일근로를 하여 받는 급여 러. 근로자가 사내급식이나 이와 유사한 방법으로 제공받는 식사 기타 음식물 또는 근로자(식사 기타 음식물을 제공받지 아니하는 자에 한정한다)가 받는 월 20만원 이하의 식사대 머. 근로자 또는 그 배우자의 출산이나 자녀의 보육과 관련하여 사용자로부터 지급받는 다음의 급여 1) 근로자(사용자와 대통령령으로 정하는 특수관계에 있는 자는 제외한다) 또는 그 배우자의 출산과 관련하여 자녀의 출생일 이후 2년 이내에 사용자로부터 대통령령으로 정하는 바에 따라 최대 두 차례에 걸쳐 지급받는 급여(2021년 1월 1일 이후 출생한 자녀에 대하여 2024년 1월 1일부터 2024년 12월 31일 사이에 지급받은 급여를 포함한다) 전액 2) 근로자 또는 그 배우자의 해당 과세기간 개시일을 기준으로 6세 이하(6세가 되는 날과 그 이전 기간을 말한다. 이하 이 조 및 제59조의4에서 같다)인 자녀의 보육과 관련하여 사용자로부터 지급받는 급여로서 월 20만원 이내의 금액 버. 「국군포로의 송환 및 대우 등에 관한 법률」에 따른 국군포로가 받는 보수 및 퇴직일시금 서. 「교육기본법」 제28조제1항에 따라 받는 장학금 중 대학생이 근로를 대가로 지급받는 장학금(

# 5. 유사도 검색한 문서를 LLM에 질문과 같이 전달 
- retrieved_doc['documents'][0]

In [104]:
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model = 'gpt-4o-mini', # tiktoken.encoding_for_model()에 전달한 model을 사용하면 더 좋은 결과 확인 가능
    messages=[
        {'role' : "system", "content":f"""당신은 한국 소득세 전문가입니다. 
        아래의 내용을 참고해서 사용자 질문에 답변해주세요.{retrieved_doc['documents'][0]}"""},
        {"role" : "user", "content" : query}
    ]
)

In [107]:
print(response.choices[0].message.content)

연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 기본세율과 소득공제를 고려해야 합니다. 

1. **기본 세율**: 소득세는 단계별 세율이 적용됩니다. 2023년 기준으로 정리하면 아래와 같은 세율이 적용됩니다.
   - 1,200만원 이하: 6%
   - 1,200만원 초과 ~ 4,600만원 이하: 15%
   - 4,600만원 초과 ~ 8,800만원 이하: 24%

2. **소득공제**: 기본공제, 인적공제 등을 포함하여 종합소득세 신고를 하게 된다면 관련된 공제를 반영해야 합니다. 일반적으로 기본공제는 주민세를 포함할 수 있고, 부양가족이 있다면 추가적인 공제를 받을 수 있습니다.

연봉 5천만원을 기준으로 단순 계산하면 아래와 같습니다:

- 1,200만원까지: 1,200만원 × 6% = 72만원
- 1,200만원 ~ 4,600만원까지: (4,600만원 - 1,200만원) × 15% = 51만원
- 4,600만원 ~ 5,000만원까지: (5,000만원 - 4,600만원) × 24% = 96만원

세액을 합산하면:
- 72만원 + 51만원 + 96만원 = 219만원 

이 외에도 세액공제 등을 고려하게 되면 최종적인 소득세액은 달라질 수 있습니다. 각종 공제 항목을 고려하지 않았으므로 실제 수치는 개인의 상황에 따라 다를 수 있습니다. 정확한 세액을 계산하려면 세무사와 상담하거나 소득세 계산기를 사용하는 것이 좋습니다.


In [108]:
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model = 'gpt-4o-mini', # tiktoken.encoding_for_model()에 전달한 model을 사용하면 더 좋은 결과 확인 가능
    messages=[
        {'role' : "system", "content":f"""당신은 한국 소득세 전문가입니다."""},
        {"role" : "user", "content" : query}
    ]
)
print(response.choices[0].message.content)

2023년 한국의 소득세 계산은 다음과 같은 세율에 따라 진행됩니다. 연봉 5천만 원인 직장인의 경우, 과세표준에 따라 소득세가 산정됩니다. 

1. **과세표준 구간**:
   - 1,200만 원까지: 6%
   - 1,200만 원 초과 4,600만 원까지: 15%
   - 4,600만 원 초과 8,800만 원까지: 24%

2. **소득세 계산**:
   - 1,200만 원 이하: 1,200만 원 × 6% = 72만 원
   - 1,200만 원 초과 4,600만 원까지(3,400만 원): 3,400만 원 × 15% = 510만 원

   따라서, 5천만 원의 소득에 대한 소득세는 다음과 같이 계산됩니다:
   - 72만 원 + 510만 원 = 582만 원

3. **세액 공제 및 환급**:
   - 여기서 인적 공제나 세액 공제를 적용하면 실제 납부할 세액이 줄어들 수 있습니다. 예를 들어, 기본 인적 공제로 150만 원이 공제될 수 있습니다.

결론적으로, 연봉 5천만 원인 직장인의 소득세는 약 582만 원에서 인적 공제 등을 반영하면 더욱 줄어들 것입니다. 정확한 금액은 개인의 상황에 따라 다를 수 있습니다.
